## Tools
Tools are defined using the tool decorator

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

# os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
# os.environ["LANGCHAIN_TRACING_V2"] = "true" 
# os.environ['LANGSMITH_PROJECT'] = 'AgenticAITraining' 

os.environ["LANGFUSE_SECRET_KEY"] = str(os.getenv("LANGFUSE_SECRET_KEY"))
os.environ["LANGFUSE_PUBLIC_KEY"] = str(os.getenv("LANGFUSE_PUBLIC_KEY"))
os.environ["LANGFUSE_BASE_URL"] = str(os.getenv("LANGFUSE_BASE_URL"))

In [2]:
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler

lf = Langfuse()
print(lf)

langfuse = get_client()
handler = CallbackHandler()

c:\Data\Codes\AgenticAITraining\Lanchain_Azure_openAI\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
from langchain.tools import tool

@tool
def sq_root(x: float) -> float:
    '''Calculate the square root of a number'''
    return x**0.5

In [4]:
print(sq_root.name)
print(sq_root.description)
print(sq_root.args)
print(sq_root.return_direct)
print(sq_root.args_schema.schema())

sq_root
Calculate the square root of a number
{'x': {'title': 'X', 'type': 'number'}}
False
{'description': 'Calculate the square root of a number', 'properties': {'x': {'title': 'X', 'type': 'number'}}, 'required': ['x'], 'title': 'sq_root', 'type': 'object'}


C:\Users\ayush.p.agrawal\AppData\Local\Temp\ipykernel_27360\3448407991.py:5: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  print(sq_root.args_schema.schema())


In [5]:
print(sq_root.invoke({'x':16}))
print(sq_root.run({'x':25}))

4.0
5.0


In [6]:
dir(sq_root)

['InputType',
 'OutputType',
 '__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__or__',
 '__orig_bases__',
 '__parameters__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_

### Pydantic Tool Input

In [7]:
from pydantic import BaseModel, Field

class CalcInp(BaseModel):
    a: int = Field(description = 'First Number')
    b: int = Field(description = 'Second number')

In [8]:
@tool('Multiplication', args_schema = CalcInp, return_direct=True)
def multiply(a: int, b: int) -> int:
    '''Multiply 2 numbers'''
    return a*b

In [11]:
obj = CalcInp(a=3, b=5)
# multiply.invoke({'a':3, 'b':3})
multiply.invoke(obj.model_dump())

15

In [12]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

Multiplication
Multiply 2 numbers
{'a': {'description': 'First Number', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second number', 'title': 'B', 'type': 'integer'}}


In [13]:
from typing import Literal

@tool
def real_number_calculator(a: float, b: float, operation: Literal['add', 'subtract', 'multiply', 'divide']) -> float:
    '''Perform basic arithmatic operation'''
    print("Invoked Calculator tool")
    if operation == 'add':
        print('Invoked Addition')
        result = a+b
        print(result)
        return result
    elif operation == 'subtract':
        print('Invoked Subtract')
        result = a-b
        print(result)
        return result
    elif operation == 'multiply':
        print('Invoked Mulitplication')
        result = a*b
        print(result)
        return result
    elif operation == 'divide':
        print('Invoked Dvision')
        if b != 0:
            return a/b
        else:
            raise ValueError('Zero Division')
    else:
        raise ValueError('Invalid Operation')

In [14]:
print(real_number_calculator.name)
print(real_number_calculator.description)
print(real_number_calculator.args)

real_number_calculator
Perform basic arithmatic operation
{'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}, 'operation': {'enum': ['add', 'subtract', 'multiply', 'divide'], 'title': 'Operation', 'type': 'string'}}


In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model='gpt-5.1',
    temperature=1
)

In [23]:
# Creating the agent using the LLM 
from langchain.agents import create_agent

agent = create_agent(
    model = llm, 
    tools = [real_number_calculator],
    system_prompt = 'You are an arithmetic wizard, answer the question only by using the provided tools'
).with_config({
    "callbacks": [handler]
})

In [24]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {'messages': [HumanMessage('What is 3.243 times 22.324')]},
)

print(response['messages'][-1].content)

Invoked Calculator tool
Invoked Mulitplication
72.396732
3.243 × 22.324 = 72.396732


In [25]:
print(response['messages'][-1])

content='3.243 × 22.324 = 72.396732' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 227, 'total_tokens': 245, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 7, 'engine_ttft_ms': 35, 'engine_ttlt_ms': 160, 'pre_inference_ms': 96, 'service_tbt_ms': 7, 'service_ttft_ms': 354, 'service_ttlt_ms': 476, 'total_duration_ms': 387, 'user_visible_ttft_ms': 258}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DfMaibOKOymbqcUE3nesPU0lrODjb', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e25cd-1e4a-7bc3-9220-9b3f50a73af8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 227, 'output_tokens': 18, 'total_tokens': 245, 'inp

In [27]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='What is 3.243 times 22.324', additional_kwargs={}, response_metadata={}, id='50ad0719-b291-45f9-94a1-1fee586093be'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 179, 'total_tokens': 217, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 5, 'engine_ttft_ms': 45, 'engine_ttlt_ms': 217, 'pre_inference_ms': 100, 'service_tbt_ms': 5, 'service_ttft_ms': 339, 'service_ttlt_ms': 513, 'total_duration_ms': 415, 'user_visible_ttft_ms': 239}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DfMagWTa4I5m6MlertfRQPuCN5hWJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run

In [29]:
response = agent.invoke(
    {'messages': [HumanMessage('Hello how are you')]}
)

pprint(response)

{'messages': [HumanMessage(content='Hello how are you', additional_kwargs={}, response_metadata={}, id='5f28e844-61b1-4541-9e00-0c0799fbb167'),
              AIMessage(content='I’m doing well and ready to help. What can I do for you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 172, 'total_tokens': 199, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 8, 'engine_ttft_ms': 43, 'engine_ttlt_ms': 264, 'pre_inference_ms': 178, 'service_tbt_ms': 8, 'service_ttft_ms': 457, 'service_ttlt_ms': 670, 'total_duration_ms': 497, 'user_visible_ttft_ms': 279}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DfMeNqODLOJkZ19xCkp4n2jS1oN4m', 'service_tier': 'default', 'finish

### Using stream on the agent call

In [30]:
query = "what is 3.1125 plus 4.1234"

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

what is 3.1125 plus 4.1234
================================== Ai Message ==================================
Tool Calls:
  real_number_calculator (call_rwlhy5JmjdOWVGbzRqW3ClMl)
 Call ID: call_rwlhy5JmjdOWVGbzRqW3ClMl
  Args:
    a: 3.1125
    b: 4.1234
    operation: add
Invoked Calculator tool
Invoked Addition
7.2359
================================= Tool Message =================================
Name: real_number_calculator

7.2359
================================== Ai Message ==================================

3.1125 plus 4.1234 equals 7.2359.


In [31]:
for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="updates"):
    print(event)

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 181, 'total_tokens': 221, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 4, 'engine_ttft_ms': 34, 'engine_ttlt_ms': 205, 'pre_inference_ms': 92, 'service_tbt_ms': 4, 'service_ttft_ms': 333, 'service_ttlt_ms': 503, 'total_duration_ms': 412, 'user_visible_ttft_ms': 241}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DfMeWUGAMWGOBavU5gUzWxh0GGhNN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e25d0-b6b3-7620-a66a-1c8e860784e7-0', tool_calls=[{'name': 'real_number_calculator', 'args': {'a': 3.1125, 'b': 4.1234, 'operation': 'add'}

In [32]:
for event in agent.stream(
    {"messages": [{"role": "user", "content": "what is 3 multiply with four"}]},
    stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

what is 3 multiply with four
================================== Ai Message ==================================
Tool Calls:
  real_number_calculator (call_cKMPKj95Y4GrTEFw8yyKSd6U)
 Call ID: call_cKMPKj95Y4GrTEFw8yyKSd6U
  Args:
    a: 3
    b: 4
    operation: multiply
Invoked Calculator tool
Invoked Mulitplication
12.0
================================= Tool Message =================================
Name: real_number_calculator

12.0
================================== Ai Message ==================================

3 multiplied by 4 is 12.


parse_docstring=True -> LangChain gets the infromation from the tool docstring

In [34]:
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_num_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮 Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [35]:
agent = create_agent(
    model = llm, 
    tools = [real_num_calculator],
    system_prompt="You are an arithmetic wizard."
)

In [36]:
result = agent.invoke({'messages': [{'role':'user', 'content': 'what is 3.5 times 2.6'}]})
print(result["messages"][-1].content)

🧮 Invoking calculator tool
3.5 times 2.6 equals 9.1.


In [37]:
for msg in result['messages']:
    msg.pretty_print()

================================ Human Message =================================

what is 3.5 times 2.6
================================== Ai Message ==================================
Tool Calls:
  calculator (call_fa19OgvTinF2lwwRbtBLpWWd)
 Call ID: call_fa19OgvTinF2lwwRbtBLpWWd
  Args:
    a: 3.5
    b: 2.6
    operation: multiply
================================= Tool Message =================================
Name: calculator

9.1
================================== Ai Message ==================================

3.5 times 2.6 equals 9.1.
